# Neuroevolution for Tic-Tac-Toe

## Learning Objectives

In this notebook, we'll explore **neuroevolution** - using evolutionary algorithms to train neural networks - by building an agent that learns to play tic-tac-toe.

By the end of this notebook, you will understand:
1. How to implement a tic-tac-toe game engine
2. How to build a simple neural network from scratch using NumPy
3. How genetic algorithms work (selection, mutation, fitness)
4. How to evolve a population of neural networks to play games
5. How neuroevolution compares to gradient-based learning

## What is Neuroevolution?

**Neuroevolution** combines neural networks with evolutionary algorithms. Instead of using backpropagation and gradient descent, we:
1. Create a **population** of neural networks with random weights
2. Evaluate their **fitness** (how well they play the game)
3. **Select** the best performers
4. Create **offspring** by mutating their weights
5. Repeat for many generations

### Why Evolution Instead of Backpropagation?

For game playing, neuroevolution has some advantages:
- **No gradient needed**: Games have discrete actions and sparse rewards
- **Exploration**: Evolution naturally explores diverse strategies
- **Simplicity**: No need for complex RL algorithms or reward shaping
- **Historical significance**: One of the earliest approaches to game AI

However, it's generally less sample-efficient than modern deep RL methods.

In [ ]:
# Import libraries
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple, Optional
from dataclasses import dataclass
import copy

# Set random seed for reproducibility
np.random.seed(42)

## Part 1: Tic-Tac-Toe Game Engine

First, we need to implement the game itself. We'll represent the board as a 3x3 grid where:
- `0` = empty square
- `1` = X (first player)
- `-1` = O (second player)

The game engine needs to:
- Track the board state
- Validate moves
- Detect wins, losses, and draws
- Provide a simple way to play games

In [ ]:
class TicTacToe:
    """Tic-Tac-Toe game engine."""
    
    def __init__(self):
        """Initialize empty board."""
        self.board = np.zeros((3, 3), dtype=np.int8)
        self.current_player = 1  # 1 for X, -1 for O
        
    def reset(self):
        """Reset the game to initial state."""
        self.board = np.zeros((3, 3), dtype=np.int8)
        self.current_player = 1
        return self.board.copy()
    
    def get_valid_moves(self) -> List[int]:
        """Get list of valid move indices (0-8)."""
        return [i for i in range(9) if self.board.flat[i] == 0]
    
    def make_move(self, position: int) -> bool:
        """
        Make a move at the given position (0-8).
        Returns True if move was valid, False otherwise.
        """
        if position < 0 or position >= 9:
            return False
        
        row, col = position // 3, position % 3
        
        if self.board[row, col] != 0:
            return False
        
        self.board[row, col] = self.current_player
        self.current_player *= -1  # Switch player
        return True
    
    def check_winner(self) -> Optional[int]:
        """
        Check if there's a winner.
        Returns: 1 if X wins, -1 if O wins, 0 if draw, None if game ongoing.
        """
        # Check rows
        for row in range(3):
            if abs(self.board[row].sum()) == 3:
                return self.board[row, 0]
        
        # Check columns
        for col in range(3):
            if abs(self.board[:, col].sum()) == 3:
                return self.board[0, col]
        
        # Check diagonals
        if abs(self.board.trace()) == 3:
            return self.board[0, 0]
        if abs(np.fliplr(self.board).trace()) == 3:
            return self.board[0, 2]
        
        # Check if board is full (draw)
        if len(self.get_valid_moves()) == 0:
            return 0
        
        # Game ongoing
        return None
    
    def render(self):
        """Display the board in a readable format."""
        symbols = {0: '.', 1: 'X', -1: 'O'}
        print()
        for i, row in enumerate(self.board):
            print(' ' + ' | '.join([symbols[cell] for cell in row]))
            if i < 2:
                print(' ---------')
        print()

# Test the game engine
game = TicTacToe()
print("Initial board:")
game.render()

# Play some moves
moves = [4, 0, 2, 1, 6]  # Center, top-left, top-right, top-middle, bottom-left
for move in moves:
    game.make_move(move)
    game.render()
    winner = game.check_winner()
    if winner is not None:
        if winner == 1:
            print("X wins!")
        elif winner == -1:
            print("O wins!")
        else:
            print("Draw!")
        break

## Part 2: Neural Network from Scratch

Now we'll build a simple feedforward neural network using only NumPy. This network will:
- Take the board state as input (18 values: 9 for X positions, 9 for O positions)
- Have one hidden layer with tanh activation
- Output 9 values (one for each square) representing move preferences

### Network Architecture

```
Input Layer (18)  →  Hidden Layer (36)  →  Output Layer (9)
  [board state]        [tanh]               [move scores]
```

### Why This Architecture?

- **18 inputs**: Separating X and O positions gives the network a clear view of who owns what
- **36 hidden nodes**: Gives enough capacity to learn patterns without being too complex
- **9 outputs**: One score per board position; higher scores = more preferred moves
- **tanh activation**: Keeps activations bounded, works well with evolution

In [ ]:
class NeuralNetwork:
    """Simple feedforward neural network for tic-tac-toe."""
    
    def __init__(self, input_size: int = 18, hidden_size: int = 36, output_size: int = 9):
        """
        Initialize network with random weights.
        
        Args:
            input_size: Number of input nodes (default 18 for tic-tac-toe)
            hidden_size: Number of hidden layer nodes
            output_size: Number of output nodes (default 9 for tic-tac-toe)
        """
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        
        # Xavier initialization for better starting weights
        self.W1 = np.random.randn(input_size, hidden_size) * np.sqrt(2.0 / input_size)
        self.b1 = np.zeros(hidden_size)
        
        self.W2 = np.random.randn(hidden_size, output_size) * np.sqrt(2.0 / hidden_size)
        self.b2 = np.zeros(output_size)
    
    def forward(self, x: np.ndarray) -> np.ndarray:
        """
        Forward pass through the network.
        
        Args:
            x: Input vector of shape (input_size,)
            
        Returns:
            Output vector of shape (output_size,)
        """
        # Hidden layer with tanh activation
        hidden = np.tanh(x @ self.W1 + self.b1)
        
        # Output layer (no activation - we'll handle it later)
        output = hidden @ self.W2 + self.b2
        
        return output
    
    def get_weights(self) -> np.ndarray:
        """Flatten all weights into a single vector (for evolution)."""
        return np.concatenate([
            self.W1.flatten(),
            self.b1.flatten(),
            self.W2.flatten(),
            self.b2.flatten()
        ])
    
    def set_weights(self, weights: np.ndarray):
        """Set weights from a flattened vector."""
        idx = 0
        
        # Restore W1
        W1_size = self.input_size * self.hidden_size
        self.W1 = weights[idx:idx + W1_size].reshape(self.input_size, self.hidden_size)
        idx += W1_size
        
        # Restore b1
        self.b1 = weights[idx:idx + self.hidden_size]
        idx += self.hidden_size
        
        # Restore W2
        W2_size = self.hidden_size * self.output_size
        self.W2 = weights[idx:idx + W2_size].reshape(self.hidden_size, self.output_size)
        idx += W2_size
        
        # Restore b2
        self.b2 = weights[idx:idx + self.output_size]
    
    def copy(self) -> 'NeuralNetwork':
        """Create a deep copy of this network."""
        new_net = NeuralNetwork(self.input_size, self.hidden_size, self.output_size)
        new_net.set_weights(self.get_weights().copy())
        return new_net

# Test the network
net = NeuralNetwork()
test_input = np.random.randn(18)
output = net.forward(test_input)
print(f"Network architecture: {net.input_size} -> {net.hidden_size} -> {net.output_size}")
print(f"Total parameters: {len(net.get_weights())}")
print(f"Sample output: {output}")

## Part 3: State Encoding

We need to convert the tic-tac-toe board into a format the neural network can understand.

### Encoding Strategy: Two-Hot Representation

We use 18 binary values:
- First 9 values: `1` if X is in that position, `0` otherwise
- Last 9 values: `1` if O is in that position, `0` otherwise

Example:
```
Board:     X | O | .        
           ---------     
           . | X | .     
           ---------     
           . | . | O     

X positions: [1,0,0, 0,1,0, 0,0,0]  (positions 0 and 4)
O positions: [0,1,0, 0,0,0, 0,0,1]  (positions 1 and 8)
Combined:    [1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1]
```

This encoding:
- Clearly separates the two players
- Is easy for the network to interpret
- Maintains spatial information (position indices)

In [ ]:
def encode_board(board: np.ndarray, player: int) -> np.ndarray:
    """
    Encode the board state from the perspective of the given player.
    
    Args:
        board: 3x3 board array
        player: 1 for X, -1 for O
        
    Returns:
        18-element vector: [own pieces (9), opponent pieces (9)]
    """
    flat_board = board.flatten()
    
    # Encode from player's perspective
    own_pieces = (flat_board == player).astype(np.float32)
    opp_pieces = (flat_board == -player).astype(np.float32)
    
    return np.concatenate([own_pieces, opp_pieces])

# Test encoding
game = TicTacToe()
game.make_move(0)  # X at top-left
game.make_move(4)  # O at center
game.make_move(8)  # X at bottom-right

print("Board:")
game.render()

print("Encoding from X's perspective (player=1):")
encoding_x = encode_board(game.board, 1)
print(f"X positions: {encoding_x[:9].astype(int)}")
print(f"O positions: {encoding_x[9:].astype(int)}")

print("\nEncoding from O's perspective (player=-1):")
encoding_o = encode_board(game.board, -1)
print(f"O positions (own): {encoding_o[:9].astype(int)}")
print(f"X positions (opp): {encoding_o[9:].astype(int)}")

## Part 4: Agent Decision Making

Now we need to connect our neural network to the game. The agent needs to:
1. Encode the current board state
2. Pass it through the network to get move scores
3. Mask out invalid moves (occupied squares)
4. Select a move (highest score)

### Move Selection Strategy

We'll use a simple approach:
- Set invalid moves to a very negative score
- Pick the move with the highest remaining score
- If all valid moves have the same score, pick randomly

In [ ]:
class NeuralAgent:
    """Agent that uses a neural network to play tic-tac-toe."""
    
    def __init__(self, network: NeuralNetwork):
        self.network = network
    
    def select_move(self, game: TicTacToe) -> int:
        """
        Select a move based on network output.
        
        Args:
            game: Current game state
            
        Returns:
            Move index (0-8)
        """
        # Encode board from current player's perspective
        state = encode_board(game.board, game.current_player)
        
        # Get network output
        move_scores = self.network.forward(state)
        
        # Mask invalid moves with very negative scores
        valid_moves = game.get_valid_moves()
        masked_scores = np.full(9, -np.inf)
        masked_scores[valid_moves] = move_scores[valid_moves]
        
        # Select move with highest score
        # (argmax will break ties randomly due to floating point noise)
        return np.argmax(masked_scores)

# Test the agent
game = TicTacToe()
agent = NeuralAgent(NeuralNetwork())

print("Agent playing 3 moves on an empty board:\n")
for i in range(3):
    move = agent.select_move(game)
    game.make_move(move)
    print(f"Move {i+1}: Position {move}")
    game.render()

## Part 5: Playing Games

We need a function to play a complete game between two agents (or an agent and a baseline opponent).

This will be used for:
- **Fitness evaluation**: How well does an agent play?
- **Testing**: Evaluate the best evolved agent
- **Visualization**: Watch evolved strategies in action

In [ ]:
def play_game(agent1: NeuralAgent, agent2: NeuralAgent, verbose: bool = False) -> int:
    """
    Play a game between two agents.
    
    Args:
        agent1: First agent (plays X)
        agent2: Second agent (plays O)
        verbose: If True, print game progress
        
    Returns:
        1 if agent1 wins, -1 if agent2 wins, 0 if draw
    """
    game = TicTacToe()
    agents = {1: agent1, -1: agent2}
    
    if verbose:
        print("Starting game...")
        game.render()
    
    while True:
        # Current agent selects move
        current_agent = agents[game.current_player]
        move = current_agent.select_move(game)
        game.make_move(move)
        
        if verbose:
            player_name = 'X' if game.current_player == -1 else 'O'  # current_player already switched
            print(f"{player_name} plays position {move}")
            game.render()
        
        # Check for game end
        result = game.check_winner()
        if result is not None:
            if verbose:
                if result == 1:
                    print("Agent 1 (X) wins!")
                elif result == -1:
                    print("Agent 2 (O) wins!")
                else:
                    print("Draw!")
            return result

# Test by playing two random agents against each other
agent1 = NeuralAgent(NeuralNetwork())
agent2 = NeuralAgent(NeuralNetwork())

result = play_game(agent1, agent2, verbose=True)

## Part 6: Random Baseline Agent

To evaluate our evolved agents, we need a baseline opponent. A random player is the simplest baseline:
- Randomly picks among valid moves
- Easy to beat for even moderately intelligent agents
- Useful for early generations

In [ ]:
class RandomAgent:
    """Agent that plays random valid moves."""
    
    def select_move(self, game: TicTacToe) -> int:
        """Select a random valid move."""
        valid_moves = game.get_valid_moves()
        return np.random.choice(valid_moves)

# Test random vs neural agent
neural_agent = NeuralAgent(NeuralNetwork())
random_agent = RandomAgent()

print("Neural agent (X) vs Random agent (O):\n")
result = play_game(neural_agent, random_agent, verbose=True)

## Part 7: Genetic Algorithm Theory

Now we're ready to implement evolution! Let's understand the key concepts:

### Key Concepts

1. **Population**: A group of neural networks (agents) that compete
2. **Genotype**: The network weights (DNA of the agent)
3. **Phenotype**: The agent's behavior in games (how it plays)
4. **Fitness**: How well an agent performs (win rate)
5. **Selection**: Choosing which agents reproduce
6. **Mutation**: Random changes to offspring weights
7. **Generation**: One cycle of evaluation → selection → reproduction

### Evolution Loop

```
1. Initialize population with random weights
2. For each generation:
   a. Evaluate fitness (play games)
   b. Select best performers (tournament selection)
   c. Create offspring by mutating parents
   d. Keep elite (best agents survive unchanged)
3. Return the best agent found
```

### Hyperparameters

- **Population size**: How many agents per generation (e.g., 50)
- **Mutation rate**: Probability of mutating each weight (e.g., 0.1)
- **Mutation strength**: Size of random perturbation (e.g., 0.3)
- **Elite count**: Number of best agents to keep unchanged (e.g., 2)
- **Tournament size**: How many agents compete in selection (e.g., 3)

## Part 8: Fitness Evaluation

Fitness measures how well an agent plays. We'll evaluate each agent by:
- Playing games against a random opponent
- Awarding points: **+1** for win, **+0.5** for draw, **-1** for loss
- Playing multiple games to get a reliable estimate

### Why These Fitness Values?

- **Wins (+1)**: Strongly rewarded - this is the goal
- **Draws (+0.5)**: Moderately rewarded - better than losing
- **Losses (-1)**: Penalized - we want to avoid these

### Number of Evaluation Games

More games = more reliable fitness estimate, but slower evolution. We'll use:
- **10 games as X** (going first)
- **10 games as O** (going second)

This ensures agents learn to play well in both positions.

In [ ]:
def evaluate_fitness(agent: NeuralAgent, num_games: int = 10) -> float:
    """
    Evaluate agent fitness by playing against a random opponent.
    
    Args:
        agent: Agent to evaluate
        num_games: Number of games to play in each position
        
    Returns:
        Fitness score (higher is better)
    """
    random_opponent = RandomAgent()
    score = 0.0
    
    # Play as first player (X)
    for _ in range(num_games):
        result = play_game(agent, random_opponent, verbose=False)
        if result == 1:  # Agent wins
            score += 1.0
        elif result == 0:  # Draw
            score += 0.5
        # Loss: score -= 1.0 (but we skip this to keep score non-negative)
    
    # Play as second player (O)
    for _ in range(num_games):
        result = play_game(random_opponent, agent, verbose=False)
        if result == -1:  # Agent wins
            score += 1.0
        elif result == 0:  # Draw
            score += 0.5
    
    return score

# Test fitness evaluation
agent = NeuralAgent(NeuralNetwork())
fitness = evaluate_fitness(agent, num_games=20)
print(f"Agent fitness: {fitness:.1f} / 40.0 possible")
print(f"Equivalent win rate: {fitness / 40.0 * 100:.1f}%")
print(f"(Random player would score ~15.0, or 37.5%)")

## Part 9: Selection and Mutation

### Tournament Selection

Tournament selection is a simple and effective selection method:
1. Randomly pick K agents (e.g., K=3)
2. The agent with highest fitness wins
3. Return the winner

This creates **selection pressure**: Better agents are more likely to reproduce.

### Mutation

Mutation introduces variation by randomly perturbing weights:
- For each weight, with probability `mutation_rate`, add Gaussian noise
- Noise ~ N(0, mutation_strength)

Mutation allows exploration of new strategies.

In [ ]:
def tournament_selection(population: List[NeuralAgent], 
                        fitness_scores: List[float], 
                        tournament_size: int = 3) -> NeuralAgent:
    """
    Select an agent using tournament selection.
    
    Args:
        population: List of agents
        fitness_scores: Fitness of each agent
        tournament_size: Number of agents to compete
        
    Returns:
        Selected agent (the winner)
    """
    # Randomly select tournament participants
    indices = np.random.choice(len(population), size=tournament_size, replace=False)
    
    # Find the best among them
    tournament_fitness = [fitness_scores[i] for i in indices]
    winner_idx = indices[np.argmax(tournament_fitness)]
    
    return population[winner_idx]

def mutate(network: NeuralNetwork, mutation_rate: float = 0.1, mutation_strength: float = 0.3) -> NeuralNetwork:
    """
    Create a mutated copy of a network.
    
    Args:
        network: Network to mutate
        mutation_rate: Probability of mutating each weight
        mutation_strength: Standard deviation of mutation noise
        
    Returns:
        Mutated network (new copy)
    """
    # Copy the network
    offspring = network.copy()
    weights = offspring.get_weights()
    
    # Apply mutations
    mutation_mask = np.random.random(len(weights)) < mutation_rate
    mutations = np.random.normal(0, mutation_strength, len(weights))
    weights[mutation_mask] += mutations[mutation_mask]
    
    offspring.set_weights(weights)
    return offspring

# Test selection and mutation
population = [NeuralAgent(NeuralNetwork()) for _ in range(5)]
fitness_scores = [1.0, 3.0, 2.0, 5.0, 4.0]

print("Testing tournament selection:")
for i in range(5):
    selected = tournament_selection(population, fitness_scores, tournament_size=3)
    selected_idx = population.index(selected)
    print(f"  Selected agent {selected_idx} (fitness={fitness_scores[selected_idx]})")

print("\nTesting mutation:")
original = NeuralNetwork()
original_weights = original.get_weights()
mutated = mutate(original, mutation_rate=0.2, mutation_strength=0.5)
mutated_weights = mutated.get_weights()
num_changed = np.sum(original_weights != mutated_weights)
print(f"  {num_changed} / {len(original_weights)} weights changed")
print(f"  Change rate: {num_changed / len(original_weights) * 100:.1f}%")

## Part 10: Evolution Loop

Now we can implement the main evolution algorithm!

### Algorithm Steps

```python
for generation in range(num_generations):
    # 1. Evaluate fitness of all agents
    fitness_scores = [evaluate_fitness(agent) for agent in population]
    
    # 2. Track statistics
    best_fitness = max(fitness_scores)
    avg_fitness = mean(fitness_scores)
    
    # 3. Select elite (best agents survive unchanged)
    elite = top_k_agents(population, fitness_scores, k=elite_count)
    
    # 4. Create offspring to fill population
    offspring = []
    while len(offspring) < population_size - elite_count:
        parent = tournament_selection(population, fitness_scores)
        child = mutate(parent)
        offspring.append(child)
    
    # 5. New population = elite + offspring
    population = elite + offspring
```

### Elitism

Elitism ensures the best agents survive unchanged to the next generation. This:
- Prevents losing the best solution found so far
- Speeds up convergence
- Is standard practice in genetic algorithms

In [ ]:
@dataclass
class EvolutionStats:
    """Statistics for one generation."""
    generation: int
    best_fitness: float
    avg_fitness: float
    worst_fitness: float

def evolve_population(population_size: int = 50,
                     num_generations: int = 100,
                     elite_count: int = 2,
                     tournament_size: int = 3,
                     mutation_rate: float = 0.1,
                     mutation_strength: float = 0.3,
                     eval_games: int = 10) -> Tuple[NeuralAgent, List[EvolutionStats]]:
    """
    Evolve a population of neural network agents.
    
    Args:
        population_size: Number of agents per generation
        num_generations: Number of generations to evolve
        elite_count: Number of best agents to keep unchanged
        tournament_size: Size of tournament for selection
        mutation_rate: Probability of mutating each weight
        mutation_strength: Standard deviation of mutation
        eval_games: Number of games per agent for fitness evaluation
        
    Returns:
        (best_agent, statistics_per_generation)
    """
    # Initialize population
    print(f"Initializing population of {population_size} agents...")
    population = [NeuralAgent(NeuralNetwork()) for _ in range(population_size)]
    
    stats_history = []
    best_agent = None
    best_fitness_ever = -np.inf
    
    print(f"\nStarting evolution for {num_generations} generations...\n")
    
    for gen in range(num_generations):
        # Evaluate fitness
        fitness_scores = [evaluate_fitness(agent, num_games=eval_games) for agent in population]
        
        # Track statistics
        best_fitness = max(fitness_scores)
        avg_fitness = np.mean(fitness_scores)
        worst_fitness = min(fitness_scores)
        
        stats = EvolutionStats(gen, best_fitness, avg_fitness, worst_fitness)
        stats_history.append(stats)
        
        # Update best agent ever
        if best_fitness > best_fitness_ever:
            best_fitness_ever = best_fitness
            best_idx = fitness_scores.index(best_fitness)
            best_agent = population[best_idx]
        
        # Print progress
        if gen % 10 == 0 or gen == num_generations - 1:
            print(f"Gen {gen:3d}: Best={best_fitness:.1f}, Avg={avg_fitness:.1f}, Worst={worst_fitness:.1f}")
        
        # Sort population by fitness
        sorted_indices = np.argsort(fitness_scores)[::-1]  # Descending order
        population = [population[i] for i in sorted_indices]
        fitness_scores = [fitness_scores[i] for i in sorted_indices]
        
        # Elite: keep best agents
        elite = [NeuralAgent(agent.network.copy()) for agent in population[:elite_count]]
        
        # Create offspring
        offspring = []
        while len(offspring) < population_size - elite_count:
            # Select parent via tournament
            parent = tournament_selection(population, fitness_scores, tournament_size)
            
            # Create mutated offspring
            child_network = mutate(parent.network, mutation_rate, mutation_strength)
            offspring.append(NeuralAgent(child_network))
        
        # New population
        population = elite + offspring
    
    print(f"\nEvolution complete!")
    print(f"Best fitness achieved: {best_fitness_ever:.1f}")
    
    return best_agent, stats_history

## Part 11: Train the Population

Let's evolve our agents! We'll use moderate hyperparameters for a balance between speed and quality.

**Hyperparameters:**
- Population: 50 agents
- Generations: 50 (increase for better results, but slower)
- Elite: 2 best agents
- Tournament: 3 agents compete
- Mutation rate: 10% of weights
- Mutation strength: 0.3

**Expected behavior:**
- Early generations: Random play, fitness ~15
- Middle generations: Learning basic strategies, fitness ~25
- Late generations: Strong play, fitness ~30-35

This will take a few minutes to run.

In [ ]:
# Run evolution
best_agent, stats = evolve_population(
    population_size=50,
    num_generations=50,
    elite_count=2,
    tournament_size=3,
    mutation_rate=0.1,
    mutation_strength=0.3,
    eval_games=10
)

## Part 12: Visualize Evolution Progress

Let's plot the fitness over generations to see how the population improved.

In [ ]:
# Extract data
generations = [s.generation for s in stats]
best_fitness = [s.best_fitness for s in stats]
avg_fitness = [s.avg_fitness for s in stats]
worst_fitness = [s.worst_fitness for s in stats]

# Plot
plt.figure(figsize=(12, 6))
plt.plot(generations, best_fitness, label='Best', linewidth=2, color='green')
plt.plot(generations, avg_fitness, label='Average', linewidth=2, color='blue')
plt.plot(generations, worst_fitness, label='Worst', linewidth=1, color='red', alpha=0.5)

plt.xlabel('Generation', fontsize=12)
plt.ylabel('Fitness', fontsize=12)
plt.title('Evolution of Tic-Tac-Toe Agents', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nFitness improvement:")
print(f"  Initial best: {stats[0].best_fitness:.1f}")
print(f"  Final best: {stats[-1].best_fitness:.1f}")
print(f"  Improvement: {stats[-1].best_fitness - stats[0].best_fitness:.1f} (+{(stats[-1].best_fitness / stats[0].best_fitness - 1) * 100:.1f}%)")

## Part 13: Evaluate the Best Agent

Let's thoroughly test our best evolved agent:
1. Play many games against a random opponent
2. Calculate win/draw/loss percentages
3. Compare to a random baseline (~37% win rate expected)

In [ ]:
def detailed_evaluation(agent: NeuralAgent, num_games: int = 100) -> dict:
    """
    Detailed evaluation of an agent.
    
    Returns:
        Dictionary with wins, draws, losses counts and percentages
    """
    random_opponent = RandomAgent()
    
    wins_as_x = draws_as_x = losses_as_x = 0
    wins_as_o = draws_as_o = losses_as_o = 0
    
    # Play as X
    for _ in range(num_games):
        result = play_game(agent, random_opponent)
        if result == 1:
            wins_as_x += 1
        elif result == 0:
            draws_as_x += 1
        else:
            losses_as_x += 1
    
    # Play as O
    for _ in range(num_games):
        result = play_game(random_opponent, agent)
        if result == -1:
            wins_as_o += 1
        elif result == 0:
            draws_as_o += 1
        else:
            losses_as_o += 1
    
    total = 2 * num_games
    wins = wins_as_x + wins_as_o
    draws = draws_as_x + draws_as_o
    losses = losses_as_x + losses_as_o
    
    return {
        'wins': wins,
        'draws': draws,
        'losses': losses,
        'win_rate': wins / total,
        'draw_rate': draws / total,
        'loss_rate': losses / total,
        'wins_as_x': wins_as_x,
        'wins_as_o': wins_as_o,
        'total_games': total
    }

# Evaluate best agent
print("Evaluating best evolved agent against random opponent...\n")
results = detailed_evaluation(best_agent, num_games=200)

print(f"Results over {results['total_games']} games:")
print(f"  Wins:   {results['wins']:3d} ({results['win_rate']*100:.1f}%)")
print(f"  Draws:  {results['draws']:3d} ({results['draw_rate']*100:.1f}%)")
print(f"  Losses: {results['losses']:3d} ({results['loss_rate']*100:.1f}%)")
print(f"\nWin breakdown:")
print(f"  As X (first):  {results['wins_as_x']} / 200")
print(f"  As O (second): {results['wins_as_o']} / 200")
print(f"\nExpected random performance: ~37% wins, ~13% draws, ~50% losses")

## Part 14: Watch the Best Agent Play

Let's watch some games to see the evolved strategy in action!

In [ ]:
print("=" * 50)
print("GAME 1: Evolved Agent (X) vs Random Agent (O)")
print("=" * 50)
random_opponent = RandomAgent()
play_game(best_agent, random_opponent, verbose=True)

print("\n" + "=" * 50)
print("GAME 2: Random Agent (X) vs Evolved Agent (O)")
print("=" * 50)
play_game(random_opponent, best_agent, verbose=True)

print("\n" + "=" * 50)
print("GAME 3: Evolved Agent (X) vs Random Agent (O)")
print("=" * 50)
play_game(best_agent, random_opponent, verbose=True)

## Part 15: Strategy Visualization

Let's visualize which moves the evolved agent prefers in different situations.

We'll create heatmaps showing:
- Move preferences on an empty board
- Move preferences in various board states

In [ ]:
def visualize_move_preferences(agent: NeuralAgent, board_state: np.ndarray, title: str):
    """
    Visualize the agent's move preferences for a given board state.
    """
    game = TicTacToe()
    game.board = board_state.copy()
    game.current_player = 1  # Assume we're X
    
    # Get move scores
    state = encode_board(game.board, game.current_player)
    move_scores = agent.network.forward(state)
    
    # Mask invalid moves
    valid_moves = game.get_valid_moves()
    for i in range(9):
        if i not in valid_moves:
            move_scores[i] = np.nan
    
    # Reshape for visualization
    heatmap = move_scores.reshape(3, 3)
    
    # Plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    # Board state
    symbols = {0: '', 1: 'X', -1: 'O'}
    board_display = np.array([[symbols[cell] for cell in row] for row in board_state])
    
    ax1.imshow(np.ones((3, 3)), cmap='gray', vmin=0, vmax=1, alpha=0.2)
    for i in range(3):
        for j in range(3):
            text = board_display[i, j]
            color = 'blue' if text == 'X' else ('red' if text == 'O' else 'black')
            ax1.text(j, i, text, ha='center', va='center', fontsize=40, 
                    color=color, fontweight='bold')
    ax1.set_xticks([])
    ax1.set_yticks([])
    ax1.set_title('Board State', fontsize=14, fontweight='bold')
    for spine in ax1.spines.values():
        spine.set_edgecolor('black')
        spine.set_linewidth(2)
    
    # Move preferences
    im = ax2.imshow(heatmap, cmap='RdYlGn', vmin=np.nanmin(move_scores), vmax=np.nanmax(move_scores))
    for i in range(3):
        for j in range(3):
            value = heatmap[i, j]
            if not np.isnan(value):
                ax2.text(j, i, f'{value:.2f}', ha='center', va='center', 
                        fontsize=16, fontweight='bold')
    ax2.set_xticks([])
    ax2.set_yticks([])
    ax2.set_title('Move Scores (higher = more preferred)', fontsize=14, fontweight='bold')
    plt.colorbar(im, ax=ax2)
    
    fig.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

# Visualize on empty board
visualize_move_preferences(best_agent, np.zeros((3, 3), dtype=np.int8), 
                          "Opening Move Preferences")

# Visualize in a specific situation
board = np.array([
    [1, -1, 0],
    [0, 1, 0],
    [0, 0, -1]
], dtype=np.int8)
visualize_move_preferences(best_agent, board, 
                          "Move Preferences in Mid-Game")

# Visualize when there's a winning move
board = np.array([
    [1, 1, 0],
    [-1, -1, 0],
    [0, 0, 0]
], dtype=np.int8)
visualize_move_preferences(best_agent, board, 
                          "Can the agent spot the winning move? (position 2)")

## Part 16: Compare Early vs Late Generations

Let's compare how agents from early and late generations play to see the improvement.

In [ ]:
# Create a random (early generation-like) agent
random_net_agent = NeuralAgent(NeuralNetwork())

print("Comparing early generation (random) vs late generation (evolved)\n")

# Evaluate random network
print("Early generation (random network):")
early_results = detailed_evaluation(random_net_agent, num_games=100)
print(f"  Win rate: {early_results['win_rate']*100:.1f}%")
print(f"  Draw rate: {early_results['draw_rate']*100:.1f}%")
print(f"  Loss rate: {early_results['loss_rate']*100:.1f}%")

# Evaluate best evolved agent
print("\nLate generation (evolved network):")
late_results = detailed_evaluation(best_agent, num_games=100)
print(f"  Win rate: {late_results['win_rate']*100:.1f}%")
print(f"  Draw rate: {late_results['draw_rate']*100:.1f}%")
print(f"  Loss rate: {late_results['loss_rate']*100:.1f}%")

print("\nImprovement:")
print(f"  Win rate: +{(late_results['win_rate'] - early_results['win_rate'])*100:.1f} percentage points")
print(f"  Relative improvement: {(late_results['win_rate'] / early_results['win_rate'] - 1)*100:.1f}%")

## Part 17: Experiments and Extensions

Now that we have a working neuroevolution system, let's explore some questions:

### Reflection Questions

1. **Population size**: How does changing the population size affect:
   - Convergence speed?
   - Final performance?
   - Computation time?

2. **Mutation rate**: What happens with:
   - Very low mutation (< 0.05)?
   - Very high mutation (> 0.3)?

3. **Network architecture**: How would performance change with:
   - More hidden layers?
   - More/fewer hidden nodes?

4. **Evaluation games**: How does the number of evaluation games affect:
   - Reliability of fitness estimates?
   - Total training time?

### Try It Yourself!

Modify the hyperparameters in the evolution function and re-run training. Some suggestions:

```python
# Experiment 1: Larger population
evolve_population(population_size=100, num_generations=50)

# Experiment 2: Higher mutation
evolve_population(mutation_rate=0.2, mutation_strength=0.5)

# Experiment 3: Longer evolution
evolve_population(num_generations=200)
```

## Part 18: Advanced Topics and Extensions

### Co-evolution

Instead of evolving against a fixed opponent, we could:
- Evolve two populations simultaneously
- Each population plays against the other for fitness
- This creates an "arms race" driving both to improve

**Advantages:**
- More challenging opponents over time
- Discovers more sophisticated strategies
- More realistic competitive environment

**Challenges:**
- Can lead to cycling strategies (rock-paper-scissors dynamics)
- Harder to measure absolute progress

### Novelty Search

Instead of only rewarding wins, reward **novel behaviors**:
- Track which strategies have been discovered
- Reward agents that play differently from previous ones
- Can discover more creative strategies

### Crossover

We only used mutation. We could also implement **crossover**:
- Combine weights from two parent networks
- Methods: uniform crossover, layer-wise, or parameter-wise
- Can combine strengths of different agents

### NEAT (NeuroEvolution of Augmenting Topologies)

NEAT evolves both weights AND network architecture:
- Start with minimal networks
- Mutations can add/remove nodes and connections
- Discovers appropriate network complexity automatically
- More complex but very powerful

### Comparison to Other Methods

How does neuroevolution compare to alternatives?

**vs. Minimax (game tree search):**
- Minimax is optimal for tic-tac-toe
- Neuroevolution learns approximate strategies
- Neuroevolution scales better to complex games

**vs. Deep Reinforcement Learning:**
- RL is more sample-efficient (learns from fewer games)
- Neuroevolution is simpler (no gradients needed)
- RL works better for large state spaces
- Neuroevolution naturally explores diverse strategies

**vs. Supervised Learning:**
- Supervised needs expert game data
- Neuroevolution learns from scratch
- Supervised can imitate experts directly

## Part 19: Summary and Key Takeaways

### What We Built

1. **Tic-Tac-Toe game engine** with move validation and win detection
2. **Neural network from scratch** using only NumPy
3. **Genetic algorithm** with tournament selection and mutation
4. **Evolution system** that trains agents to play tic-tac-toe

### Key Insights

1. **No gradients needed**: Evolution works without backpropagation
2. **Population diversity matters**: Multiple agents explore different strategies
3. **Fitness landscape**: Evolution climbs toward better strategies over generations
4. **Elitism helps**: Keeping the best agents prevents losing good solutions
5. **Simple can work**: Even basic genetic algorithms can solve game problems

### Advantages of Neuroevolution

✅ Simple to implement (no gradient calculations)
✅ Works with discrete actions and sparse rewards
✅ Naturally explores diverse strategies
✅ Can evolve network architectures (with NEAT)
✅ Parallelizable (evaluate agents independently)

### Limitations

❌ Sample inefficient (needs many game evaluations)
❌ Slower than gradient-based methods for large networks
❌ Can get stuck in local optima
❌ Requires careful hyperparameter tuning
❌ Less effective than minimax for small, solvable games

### When to Use Neuroevolution

Neuroevolution is a good choice when:
- The environment has discrete actions
- Rewards are sparse or hard to shape
- You want to explore diverse strategies
- The problem is too complex for exact methods (minimax)
- But not so complex that deep RL is required

### Applications Beyond Tic-Tac-Toe

Neuroevolution has been successfully applied to:
- **Game AI**: Mario, Pac-Man, racing games
- **Robotics**: Walking controllers, drone navigation
- **Neural architecture search**: Finding optimal network designs
- **Multi-agent systems**: Cooperative and competitive behaviors

### Further Learning

To dive deeper:
1. **NEAT**: Read the original paper on topology evolution
2. **OpenAI Evolution Strategies**: Modern large-scale neuroevolution
3. **Quality Diversity**: Algorithms that maintain diverse populations
4. **Genetic Programming**: Evolving code and algorithms, not just weights

---

## Congratulations!

You've successfully implemented neuroevolution from scratch and trained an agent to play tic-tac-toe. You now understand:
- How genetic algorithms work
- How to evolve neural networks without gradients
- The trade-offs between evolution and other learning methods

This is a powerful technique with many applications. Keep experimenting! 🧬🎮